# Initialize your project folder, set simulation inputs
**Before running notebook: Create folder in your file system**\
Create a new folder for your project and put all data files you want to process in it.\
Data files are typically your exported results from Sciex Analyst saved in CSV or TXT format. Make sure you exported all columns.

**File naming conventions**\
(1) Files reflecting results from the **core method** must end with **_core.txt** or **_core.csv**\
(2) Files reflecting results from the **extended method** must end with **_extended.txt** or **extended.csv**\
(3) If you want to combine core method and extended method the **base name of the files to be combined** (string which comes before _core or _extended) must be **exactly the same**.

**Change project_folder variable in the current notebook** \
Change the variable **project_folder** in the first Code block of the notebook, so that it points to the project folder you just created.\
You can copy the path of your project folder on windows by right clicking on the folder in your file explorer and choosing the option *Copy as Path*. \
The same works on Mac OS: Control-click or right-click on the folder in Finder. Press the Option (Alt) key. Choose *Copy [foldername] as Pathname*.

**Change eis_identifier variable in the current notebook** \
Change the variable **eis_identifier** in the first Code block of the notebook. The variable is used to identify extracted internal standards (EIS)
from the Compound Names in the raw data. For data orginating from the last calibration (before June 2025) the identifier is 'IDA', for data originating from
the new calibration (after June 2025) it is 'EIS'.

**Run the current notebook** \
After you run the notebook your project folder should be populated with two new subfolders: \
(1) code_parameters, which contains three new CSV files: (i) recovery_thresholds.csv, (ii) sample_parameters.csv, (iii) simulation_parameters.csv. \
(2) processed_data, which contains one subdirectory: plots \
Make sure the directories have been created

**Change your code parameters** \
(i) Adopt the recovery threshold in recovery_thresholds.csv values to the conditions of your experiments and save the changes. \
(ii) Open sample_parameters.csv and change the sample names (column **alternative name (used in results)**) in case your internal sample naming conventions are not a good choice for sharing your resuls.
The sample names will appear in your final result tables. Sample number and sample ID are used for internal linking purposes, do not change them.\
Provide sample size (column **volume, weight, number of samples, etc.**) and sample unit (column: **unit (e.g. g/mL/sample)**) for each sample.\
Change the column **used for mdl calculation** to TRUE for all blank samples you want to use for the calculation of method detection limits (MDL). \
In case you are working with passive samplers: enter **temperature in C** and **deployment time in days**. Leave the columns blank if you are not working with passive samplers. \
Change the **dilution factor**, in case it was not one. \
Save the changes and close simulation_parameters.csv. \
(iii) Change the simulation parameters in.....

**Run data_analysis.ipynb notebook**

In [ ]:
# relevant input variables
project_folder = r'test'

# used to identify extracted internal standards (EIS), previously known as IDA
eis_identifier = 'IDA'
nis_identifier = 'IPS'

# used to identify internal standards from column "Component Name" - usually you do not have to change it
standard_identifiers = 'EIS|NIS|IDA|IPS|13C|d-|d3-|d5-|18O'

# used to identify which channel to use when duplicates of compounds are available due to the combination of core and extended method - usually you don't have to change it
# Set parameter to core if you want to use the channel from the core method for further caluclations.
# Set parameter to extended if you want to use the channel from the extended method for further calculations.
# Set parameter to average if you want to use the average of both channels for further calculations.
channel_selection = 'core'

In [ ]:
# import necessary packages
import pandas as pd
import os

# import functions from utils.py
from utils import (read_in_data_files, get_sample_id_and_name, clean_up_data, get_compounds_and_standards)

# display settings
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_colwidth', None)  # Show full width of columns

In [ ]:
# create directory for output data, thow error if it already exists
if os.path.isdir(os.path.join(project_folder, 'processed_data')):
    raise ImportError("""
        Processed Data Folder already exists in your project folder.
        The script does not create a new one.            
                      """)
os.makedirs(os.path.join(project_folder, 'processed_data'))
os.makedirs(os.path.join(project_folder, 'processed_data', 'plots'))

In [ ]:
# create directory for code parameters, thow error if it already exists
if os.path.isdir(os.path.join(project_folder, 'code_parameters')):
    raise ImportError("""
        Simulation Parameters Folder already exists in your project folder.
        The script does not create a new one.            
                      """)
os.makedirs(os.path.join(project_folder, 'code_parameters'))

In [ ]:
### Create csv file sample_parameters.csv for sample inputs.
# Information about available samples, LCMS Code and internal names is extracted from raw data.

# combine all data files to one common dataframe
data = read_in_data_files(project_folder)

# extract list of samples from raw data
sample_list = get_sample_id_and_name(data)
display(sample_list)

# for simplicity: get rid of standard samples in sample list
visible_sample_list = sample_list.loc[sample_list['Sample Type'] != 'Standard', :]

# create a sample input parameter file
sample_input_data = pd.DataFrame(columns=[
    'batch name', 'sample id', 'alternative name (used in results)', 
    'volume/weight/number of samples', 'unit (e.g. g/mL/sample)', 'used for mdl calculation',
    'temperature in C', 'deployment time in days', 'dilution factor',
    ],
    index=visible_sample_list.index
    )
# fill sample id column
sample_input_data['sample id'] = visible_sample_list['Sample ID']
sample_input_data['batch name'] = visible_sample_list['Batch Name']
# fill sample names column and get rid of 'Core' and 'Ext' ending of sample names, as Core and Extended method will be combined by the code internally.
sample_names = []
for (sample_name_core, sample_name_extended) in zip(visible_sample_list['Sample Name Core'], visible_sample_list['Sample Name Extended']):
    if pd.isnull(sample_name_extended):
        sample_names.append(sample_name_core[:-5])
    else:
        sample_names.append(sample_name_extended[:-4])
sample_input_data['alternative name (used in results)'] = sample_names

# set default values for remaining columns
sample_input_data['volume/weight/number of samples'] = 1
sample_input_data['used for mdl calculation'] = False
sample_input_data['unit (e.g. g/mL/sample)'] = 'sample'
sample_input_data['dilution factor'] = 1

# write dataframe to csv
sample_input_data.to_csv(path_or_buf=str(os.path.join(project_folder, 'code_parameters', 'sample_parameters.csv')), index=True)

In [ ]:
# Information about available target analystes, available extracted internal standards (EIS), and available non-extracted internal standards (NIS), is extracted from raw data.

# clean up data first (misleading compound names are corrected)
data = clean_up_data(data=data, sample_list=sample_list, channel_selection=channel_selection)

# extract list of compound names and standard names from raw data
compounds_msms, compounds_tof, eis_nis_msms, eis_nis_tof, _, _, _, _ = get_compounds_and_standards(
    data=data, sample_list=sample_list, standard_identifiers=standard_identifiers,
    eis_identifier=eis_identifier, nis_identifier=nis_identifier
    )

In [ ]:
### Create csv file recovery_or_standard_response_thresholds.csv for threshold inputs for recovery rates or internal standard response deviations.
# extract only EIS from MSMS Channel for thresholds
eis_sorted = [standard for standard in eis_nis_msms if (not pd.isnull(standard) and eis_identifier in standard)]

# create default data frame for thresholds
recovery_or_standard_response_thresholds = pd.DataFrame(columns=[
    'lower threshold [%]', 'upper threshold [%]'
    ],
    index=eis_sorted
    )
# input default thresholds
recovery_or_standard_response_thresholds['lower threshold [%]'] = 50
recovery_or_standard_response_thresholds['upper threshold [%]'] = 150
# write data frame to csv
recovery_or_standard_response_thresholds.to_csv(path_or_buf=str(os.path.join(project_folder, 'code_parameters', 'recovery_or_standard_response_thresholds.csv')), index=True)

In [ ]:
### Create csv file retention_time_and_iar_thresholds.csv for threshold inputs for accepted retention time differences and ion abundance ratio deviations
# get native PFAS, EIS and NIS from MSMS Channel for thresholds
native_pfas_eis_nis_sorted = []

# loop over pair of msms and tof compound names
for (msms_compound, tof_compound) in zip(compounds_msms + eis_nis_msms, compounds_tof + eis_nis_tof):
    # use msms compound if available
    if not pd.isnull(msms_compound):
        native_pfas_eis_nis_sorted.append(msms_compound)
    # get compound name from tof channel in case msms is not available
    else:
        native_pfas_eis_nis_sorted.append(tof_compound[:-7])

# create default data frame for accepted retention time differences and ion abundance ratio deviations
retention_time_and_iar_thresholds = pd.DataFrame(columns=[
    'accepted retention time difference [min]', 'allowed ion abundance ratio deviation [%]'
    ],
    index=native_pfas_eis_nis_sorted
    )
# input default thresholds
retention_time_and_iar_thresholds['accepted retention time difference [min]'] = 0.1
retention_time_and_iar_thresholds['accepted ion abundance ratio deviation [%]'] = 50

# set threshold to 0.4 minutes if corresponding EIS has different chemical structure
# loop over pfas compounds
for pfas_compound in native_pfas_eis_nis_sorted:
    # skip standards
    if eis_identifier in pfas_compound or nis_identifier in pfas_compound:
        continue
    # check if any EIS matches names of PFAS compound -> means exact chemical structure is available
    # and skip iteration of loop if it is available
    if len([1 for eis in eis_sorted if pfas_compound in eis]):
        continue
    # set the retention time threshold to 0.4 minutes if exact EIS name was not available.
    retention_time_and_iar_thresholds.loc[pfas_compound, 'accepted retention time difference [min]'] = 0.4

# write data frame to csv
retention_time_and_iar_thresholds.to_csv(path_or_buf=str(os.path.join(project_folder, 'code_parameters', 'retention_time_and_iar_thresholds.csv')), index=True)


In [ ]:
### Create csv file simulation_parameters.csv for inputs to the data_analysis.ipynb notebook.
# create default data frame for recovery thresholds
simulation_parameters = pd.DataFrame(columns=['Parameter Name', 'Parameter Description', 'Parameter Value'])

# Write parameters, descriptions and default values to data frame.
simulation_parameters.loc[0] = [
"EIS identifier", "Repeating substring, which is used to identify extracted internal standards (EIS), formally known as IDA, from compound name.", eis_identifier
]
simulation_parameters.loc[1] = [
"NIS identifier", "Repeating substring, which is used to identify non-extracted internal standards (NIS), formally known as IPS, from compound name.", "IPS"
]

simulation_parameters.loc[2] = [
"calibration midpoint identifier", "Repeating substring, which is used to identify calibration midpoint from sample name.", "CS6"
]

simulation_parameters.loc[3] = [
"calibration reference", "To choose reference data from the calibration (midpoint) for QAQC, you can use data from initial calibration, or data from continuous calibration verification (CCV) \n"+ \
"Set parameter to calibration if you want to use data from initial calibration as reference. \n" +\
"Set parameter to ccv if you want to use data from continuos calibration verification as reference.", "calibration"
]

simulation_parameters.loc[4] = [
"channel selection", "For some cases, when using both core and extended method, some compounds in the TOF channels are avaialbe twice within the same sample. \n"+ \
"Set parameter to core if you want to use the channel from the core method for further caluclations. \n" +\
"Set parameter to extended if you want to use the channel from the extended method for further calculations. \n" +\
"Set parameter to average if you want to use the average of both channels for further calculations.",
channel_selection
]

simulation_parameters.loc[5] = [
"long format channel selection", "Which channels should be selected to report calculated concentrations in the long format results table. \n"+ \
"Set parameter to msms if you want to report the msms channel. \n" +\
"Set parameter to tof if you want to report the tof channel. . \n" +\
"Set parameter to both if you want to report both channels.", 'msms',
]

# set parameter to index and delete column
simulation_parameters.set_index('Parameter Name', inplace=True)

# write data frame to csv
simulation_parameters.to_csv(path_or_buf=str(os.path.join(project_folder, 'code_parameters', 'simulation_parameters.csv')), index=True)